# Results Analysis

Table of Contents
- [Setup](#setup)
- [Model Results on Datasetse](#model-results-on-datasets)

In [ ]:
# Imports
from __future__ import annotations
import json
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from utils.embedding_models import EMBEDDING_MODELS

In [ ]:
# functions, incorporate into utils later
def read_macro_f1(
    run_name: str,
    reports_dir: str | Path = Path("reports"),
) -> dict[str, float]:
    """Load hold-out test macro-F1 for every model under a run.

    Looks in ``{reports_dir}/linear-probe/{run_name}`` and
    ``{reports_dir}/cnn/{run_name}``. Keys are the model directory names.
    """
    reports_dir = Path(reports_dir)
    scores: dict[str, float] = {}

    for kind in ("linear-probe", "cnn"):
        run_dir = reports_dir / kind / run_name
        if not run_dir.exists():
            continue
        for result_path in sorted(run_dir.glob("*/result.json")):
            result = json.loads(result_path.read_text(encoding="utf-8"))
            if result.get("skipped"):
                continue
            macro_f1 = result.get("test_results", {}).get("test_metrics", {}).get("macro_f1")
            if macro_f1 is None:
                continue
            scores[result_path.parent.name] = float(macro_f1)
    return scores

def plot_macro_f1_bars(
    model_scores: dict[str, float],
    *,
    title: str | None = None,
    panel_label: str | None = None,
    color: str = "#4c78a8",
    cnn_label: str = "ResNet50 CNN",
    figsize: tuple[float, float] | None = None,
    ax: plt.Axes | None = None,
    save_fp: str | Path | None = None,
) -> tuple[plt.Figure, plt.Axes]:
    """Horizontal bar plot of hold-out macro-F1, one bar per model."""
    items = sorted(model_scores.items(), key=lambda kv: kv[1])
    names = [name for name, _ in items]
    values = [score for _, score in items]
    n_models = len(names)
    if figsize is None:
        figsize = (4.5, max(3.5, 0.32 * n_models + 1.2))

    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    y = range(n_models)
    with plt.rc_context({"hatch.linewidth": 1.8}):
        for i, (name, value) in enumerate(zip(names, values)):
            is_cnn = name == cnn_label
            ax.barh(
                i,
                value,
                color=color,
                edgecolor="0.15" if is_cnn else "white",
                linewidth=0.8 if is_cnn else 0.5,
                hatch="///" if is_cnn else None,
            )
            ax.text(
                value + 0.01,
                i,
                f"{value:.3f}",
                va="center",
                ha="left",
                fontsize=8,
            )

    ax.set_yticks(list(y), names)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("Macro F1")
    ax.set_ylabel("")
    if title:
        ax.set_title(title)
    ax.grid(axis="x", alpha=0.3)

    if panel_label:
        ax.text(
            0.0,
            1.02,
            panel_label,
            transform=ax.transAxes,
            fontsize=12,
            fontweight="bold",
            va="bottom",
            ha="left",
        )

    if fig is not None:
        fig.tight_layout()
    if save_fp is not None:
        save_fp = Path(save_fp)
        save_fp.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_fp, dpi=300, bbox_inches="tight")
    return fig, ax


def _macro_f1_model_label(
    model_name: str,
    embedding_models: dict[str, dict] | None,
    cnn_dir_name: str,
    cnn_label: str,
) -> str:
    if model_name == cnn_dir_name:
        return cnn_label
    if embedding_models is not None:
        cfg = embedding_models.get(model_name)
        if cfg and cfg.get("label"):
            return str(cfg["label"])
    return model_name


def macro_f1_table(
    datasets: dict[str, dict],
    *,
    reports_dir: str | Path = Path("reports"),
    embedding_models: dict[str, dict] | None = None,
    cnn_dir_name: str = "resnet50",
    cnn_label: str = "ResNet50 CNN",
    model_col: str = "Model",
) -> pd.DataFrame:
    """Hold-out macro-F1 table: one row per model, one column per dataset.

    ``datasets`` is the notebook ``DATASETS`` mapping. Column names come from
    each entry's ``display_name`` (aβ, Tau, BACH, TIL). Models missing a
    dataset are left as NaN. The CNN row is listed first when present.
    """
    columns: list[str] = []
    by_dataset: dict[str, dict[str, float]] = {}
    for dataset in datasets.values():
        col = dataset["display_name"]
        columns.append(col)
        labeled: dict[str, float] = {}
        for model_name, score in read_macro_f1(
            dataset["run_name"], reports_dir=reports_dir
        ).items():
            labeled[
                _macro_f1_model_label(
                    model_name, embedding_models, cnn_dir_name, cnn_label
                )
            ] = score
        by_dataset[col] = labeled

    models = set().union(*(scores.keys() for scores in by_dataset.values()))
    other_models = sorted(m for m in models if m != cnn_label)
    row_order = [cnn_label] + other_models if cnn_label in models else other_models

    rows = []
    for model in row_order:
        row: dict[str, str | float | None] = {model_col: model}
        for col in columns:
            row[col] = by_dataset[col].get(model)
        rows.append(row)
    return pd.DataFrame(rows, columns=[model_col, *columns])

## Setup
<a id="setup"></a>

In [ ]:
SAVE_DIR = Path("reports/results-analysis")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "abeta": {
        "run_name": "abeta-dataset-single-label",
        "display_name": "aβ",
        "stain": "IHC",
        "domain": "neurodegeneratri",
    },
    "tau": {
        "run_name": "tau-dataset-single-label",
        "display_name": "Tau",
        "stain": "IHC",
        "domain": "neuropathology",
    },
    "bach": {
        "run_name": "bach-dataset",  # edit if the report folder uses another name
        "display_name": "BACH",
        "modality": "H&E",
        "stain": "H&E",
        "task_family": "cancer",
    },
    "till": {
        "run_name": "till-dataset",
        "display_name": "TIL",
        "modality": "H&E",
        "stain": "H&E",
        "task_family": "cancer",
    }
}

# To allow mapping model names to their labels replace forward slash with double underscore.
EMBEDDING_MODELS = {k.replace("/", "__"): v for k, v in EMBEDDING_MODELS.items()}

STAIN_BAR_COLORS = {
    "IHC": "#4c78a8",  # blue
    "H&E": "#f58518",  # orange
}

## Model Results on Datasets
<a id="model-results-on-datasets"></a>

In [ ]:
# Read the results for each dataset.
results = {}

for dataset in DATASETS.values():
    # Relabel the model names to their display names. 
    model_results = read_macro_f1(dataset["run_name"])
    
    relabeled_results = {}
    for model_name, f1 in model_results.items():
        if model_name == "resnet50":
            relabeled_results[f"ResNet50 CNN"] = f1
        else:
            relabeled_results[EMBEDDING_MODELS[model_name]["label"]] = f1

    
    results[dataset["display_name"]] = relabeled_results

macro_f1_df = macro_f1_table(DATASETS, embedding_models=EMBEDDING_MODELS)
macro_f1_df.to_csv(SAVE_DIR / "models-macro-f1.csv", index=False)
display(macro_f1_df.round(3))

for letter, dataset in zip("abcd", DATASETS.values()):
    fig, ax = plot_macro_f1_bars(
        results[dataset["display_name"]],
        figsize=(7.5,5),
        title=dataset["display_name"],
        panel_label=f"{letter})",
        color=STAIN_BAR_COLORS[dataset["stain"]],
        save_fp=SAVE_DIR / f"models-macro-f1-barplot_{dataset['display_name']}.png"
    )

In [ ]:
def per_class_f1_table(dataset: str, probe: str) -> pd.DataFrame:
    """Hold-out per-class F1 for ResNet50 CNN vs one linear probe."""
    spec = DATASETS[dataset]
    probe_dir = next(
        mid for mid, cfg in EMBEDDING_MODELS.items() if cfg["label"] == probe
    )
    reports_dir = Path("reports")

    def scores(kind: str, folder: str) -> dict[str, float]:
        path = reports_dir / kind / spec["run_name"] / folder / "result.json"
        result = json.loads(path.read_text(encoding="utf-8"))
        return {
            name: float(stats["f1"])
            for name, stats in result["test_results"]["per_label"].items()
        }

    cnn = scores("cnn", "resnet50")
    fm = scores("linear-probe", probe_dir)
    classes = list(cnn)
    return pd.DataFrame(
        [
            {"Model": "ResNet50 CNN", **{c: cnn[c] for c in classes}},
            {"Model": probe, **{c: fm[c] for c in classes}},
        ]
    )
    
per_class_f1_table("abeta", "CONCH")

## Selected Dataset Deep Dive

This section compares frozen foundation-model embeddings plus linear probes against a conventional ResNet CNN fine-tuned on `SELECTED_RUN_KEY`. Hyperparameters are selected by validation macro-F1, and the hold-out split is evaluated once. Macro-F1 is the primary score because these tile datasets are imbalanced.


In [ ]:
fig, ax, metric_df = plot_result_metrics(
    all_results,
    metrics=("macro_f1", "balanced_accuracy"),
    split="hold-out",
    class_names=CLASS_NAMES,
    title="Amyloid-β (IHC)",
    title_color="blue",
    save_dir=save_dir / "figures",
    save_name=f"{SELECTED_RUN_KEY}_holdout_metric_comparison.png",
    legend=True
)
plt.show()

In [ ]:
fig, ax, per_class_df = plot_per_class_f1(
    all_results,
    split="hold-out",
    class_names=CLASS_NAMES,
    title=f"Per-class F1 - {run_args.get('resolved_dataset', run_args.get('dataset'))}",
    save_name=f"holdout_per_class_f1-{SELECTED_RUN_KEY}.png" if save_dir else None,
    save_dir=save_dir,
)
plt.show()

In [ ]:
from ipywidgets import Dropdown, ToggleButtons, interact

_cm_results = sorted_results(all_results, split="hold-out")
_cm_model_options = [
    (f"{model_label(result)} - {method_label(result)} ({result['model_id']})", result["model_id"])
    for result in _cm_results
]


def show_confusion_matrix(
    model_id: str,
    split: str = "hold-out",
    normalize: str = "counts",
):
    normalize_arg = "true" if normalize == "row-normalized" else False
    fig, _, _ = plot_result_confusion_matrix(
        all_results,
        model_id,
        split=split,
        normalize=normalize_arg,
        class_names=CLASS_NAMES,
    )
    plt.show()


_ = interact(
    show_confusion_matrix,
    model_id=Dropdown(options=_cm_model_options, description="Model:"),
    split=ToggleButtons(
        options=[("Hold-out", "hold-out"), ("Validation", "validation")],
        description="Split:",
    ),
    normalize=ToggleButtons(
        options=[("Counts", "counts"), ("Row norm", "row-normalized")],
        description="Scale:",
    ),
)

# Optional: save one PNG per model/run without rendering a large collage.
# save_all_confusion_matrices(all_results, split="hold-out", normalize=False, save_dir=save_dir, class_names=CLASS_NAMES)
# save_all_confusion_matrices(all_results, split="hold-out", normalize="true", save_dir=save_dir, class_names=CLASS_NAMES)

In [ ]:
fig, ax, val_hold = plot_validation_vs_holdout(
    all_results,
    metric="macro_f1",
    class_names=CLASS_NAMES,
    save_name="validation_vs_holdout_macro_f1.png" if save_dir else None,
    save_dir=save_dir,
)
plt.show()

val_hold.sort_values("macro_f1_holdout", ascending=False)[
    ["model_label", "method", "macro_f1_val", "macro_f1_holdout"]
].round(3)

In [ ]:
fig, ax, compute_df = plot_compute_time_vs_performance(
    leaderboard,
    save_name="compute_time_vs_macro_f1.png" if save_dir else None,
    save_dir=save_dir,
)
plt.show()

time_cols = [
    "model_label",
    "method",
    "holdout_macro_f1",
    "ms_per_tile",
    "estimated_embedding_seconds",
    "total_training_seconds",
    "available_compute_seconds",
]
compute_table = leaderboard[[c for c in time_cols if c in leaderboard.columns]].copy()
for col in ("estimated_embedding_seconds", "total_training_seconds", "available_compute_seconds"):
    if col in compute_table.columns:
        compute_table[col] = compute_table[col].map(format_seconds)
compute_table.round(3)


## Tau Data-Scaling Analysis

This section measures how performance changes as the Tau training set grows through **1%, 5%, 10%, 25%, 50%, and 100%**. The 100% reports come from `tau-dataset-single-label`; smaller runs come from `tau-data-scaling-Mpct` under both `reports/linear-probe/` and `reports/cnn/`.

The analysis is intentionally safe to rerun while jobs are still completing: missing folders and unfinished models are reported rather than treated as zero-performing runs.

### Questions and planned views

1. **Run completeness:** Which percentages, embedding probes, and CNN results are currently available?
2. **Overall sample efficiency:** Plot hold-out macro-F1 against training percentage for every model, then a less crowded view of the strongest 100%-data probes plus the CNN.
3. **Class-specific behavior:** Track negative, pre-NFT, and iNFT F1 separately to identify classes that need more labeled data.
4. **CNN versus frozen embeddings:** Compare the CNN with the best available linear probe at each percentage and plot their performance gap.
5. **Saturation and ranking stability:** Once all runs finish, measure gains between adjacent percentages and whether model rankings stabilize as data increases.

Macro-F1 remains the primary metric because the classes are imbalanced. Balanced accuracy and per-class recall are useful secondary checks. These runs use one deterministic subset ordering, so the curves describe this nested sample sequence; repeated subset seeds would be needed for confidence intervals or statistical claims.

In [ ]:
# Load every currently available Tau scaling result.
tau_scaling_results, tau_scaling_availability, tau_scaling_frame = load_tau_scaling_results(
    class_names=CLASS_NAMES,
)
print(
    f"Loaded {len(tau_scaling_frame)} completed model/percentage results "
    f"across {tau_scaling_frame['train_percentage'].nunique() if not tau_scaling_frame.empty else 0} percentages."
)
tau_scaling_availability

### Overall learning curves

The first plot shows every completed model so missing runs remain visible as gaps. The focused plot keeps the CNN and the five linear probes with the strongest 100%-data macro-F1, which makes saturation and crossover points easier to read.

We use percentage as an ordered categorical axis rather than a linear numeric axis so the low-data regime (1–10%) is not visually compressed.

In [ ]:
if tau_scaling_frame.empty:
    print("No completed Tau scaling results are available yet.")
else:
    fig, ax, tau_all_curves = plot_tau_learning_curves(tau_scaling_frame)
    plt.show()

    full_linear = tau_scaling_frame[
        (tau_scaling_frame["train_percentage"] == 100)
        & (tau_scaling_frame["method"] == "Frozen embedding + linear probe")
    ]
    top_probe_ids = full_linear.nlargest(5, "macro_f1")["model_id"].tolist()
    cnn_ids = tau_scaling_frame.loc[
        tau_scaling_frame["method"] == "CNN fine-tune", "model_id"
    ].drop_duplicates().tolist()
    focus_model_ids = cnn_ids + top_probe_ids

    fig, ax, tau_focus_curves = plot_tau_learning_curves(
        tau_scaling_frame,
        model_ids=focus_model_ids,
        title="Tau data scaling: CNN and top five full-data embedding probes",
        figsize=(11, 6),
    )
    plt.show()


### Class-specific sample efficiency and CNN–probe gap

Overall macro-F1 can hide different data requirements across classes. The next view tracks each class for the focused models, especially whether pre-NFT or iNFT improves later than the negative class.

The companion summary compares the CNN against the best completed frozen-embedding probe at each percentage. A positive `cnn_minus_best_probe` value favors the CNN; a negative value favors the best linear probe. The identity of the best probe is retained because the winning embedding may change with data size.

In [ ]:
if tau_scaling_frame.empty:
    print("No completed Tau scaling results are available yet.")
else:
    if "focus_model_ids" not in globals():
        focus_model_ids = tau_scaling_frame["model_id"].drop_duplicates().tolist()
    fig, axes, tau_class_curves = plot_tau_per_class_f1(
        tau_scaling_frame,
        focus_model_ids,
        class_names=CLASS_NAMES,
    )
    plt.show()

    tau_method_gap = tau_cnn_probe_gap(tau_scaling_frame)
    display(tau_method_gap.round(3))

    gap_complete = tau_method_gap.dropna(subset=["best_probe_macro_f1", "cnn_macro_f1"])
    if not gap_complete.empty:
        fig, ax, _ = plot_tau_cnn_probe_gap(tau_method_gap)
        plt.show()


### Follow-up analyses after all runs finish

When the completion table is full, useful additions are:

- **Marginal gain per added data:** absolute and relative macro-F1 improvement from 1→5→10→25→50→100%, highlighting where each model saturates.
- **Fraction of full-data performance recovered:** score at each percentage divided by that model's 100% score, which compares sample efficiency despite different final ceilings.
- **Ranking stability:** Spearman correlation between the linear-probe ranking at each percentage and the 100% ranking.
- **Validation-to-hold-out generalization:** track the validation–hold-out gap across percentages to detect unstable low-data model selection.
- **Repeated-seed uncertainty:** if additional nested subset seeds are run later, plot mean curves with confidence intervals and compare area under the learning curve.

The current plots should not use missing results as zeros. Comparisons at a percentage are provisional until the availability table shows the expected number of completed linear probes and CNNs.

## Cross-Dataset Summary

This section loads every available run in `RUNS` and compares model families across datasets. Missing tau or breast reports are skipped until their result folders exist, so the same cells can be rerun as experiments land.


In [ ]:
cross_dataset_leaderboard = build_leaderboard(cross_dataset_results, class_names=CLASS_NAMES)
cross_dataset_summary = build_cross_dataset_summary(cross_dataset_leaderboard)
cnn_embedding_delta = build_cnn_vs_embedding_delta(cross_dataset_summary)

print(f"Available runs: {', '.join(analysis_runs)}")
if skipped_runs:
    print("Skipped runs:")
    for key, reason in skipped_runs.items():
        print(f"  - {key}: {reason}")

cross_display_cols = [
    "run_key",
    "dataset_display",
    "modality",
    "task_family",
    "model_id",
    "method",
    "model_family",
    "holdout_macro_f1",
    "holdout_balanced_accuracy",
    "val_macro_f1",
    "available_compute_seconds",
]
cross_display_cols = [c for c in cross_display_cols if c in cross_dataset_leaderboard.columns]
cross_dataset_table = cross_dataset_leaderboard[cross_display_cols].round(3)
maybe_save_table(cross_dataset_table, "cross_dataset_leaderboard.csv", save_dir=cross_save_dir)
cross_dataset_table


In [ ]:
summary_cols = [
    "run_key",
    "dataset_display",
    "modality",
    "task_family",
    "comparison_group",
    "model_id",
    "holdout_macro_f1",
    "holdout_balanced_accuracy",
    "available_compute_seconds",
]
summary_cols = [c for c in summary_cols if c in cross_dataset_summary.columns]
summary_table = cross_dataset_summary[summary_cols].round(3)
maybe_save_table(summary_table, "cross_dataset_best_by_family.csv", save_dir=cross_save_dir)
summary_table


In [ ]:
fig, ax, heatmap_table = plot_model_dataset_heatmap(
    cross_dataset_leaderboard,
    value_col="holdout_macro_f1",
    save_name="model_by_dataset_macro_f1_heatmap.png" if cross_save_dir else None,
    save_dir=cross_save_dir,
)
plt.show()


In [ ]:
fig, ax, best_family_plot_df = plot_best_method_by_dataset(
    cross_dataset_summary,
    save_name="best_method_family_by_dataset.png" if cross_save_dir else None,
    save_dir=cross_save_dir,
)
plt.show()


In [ ]:
if "cnn_minus_best_embedding_macro_f1" in cnn_embedding_delta.columns and cnn_embedding_delta["cnn_minus_best_embedding_macro_f1"].notna().any():
    fig, ax, delta_plot_df = plot_cnn_embedding_delta(
        cnn_embedding_delta,
        save_name="cnn_minus_best_embedding_macro_f1.png" if cross_save_dir else None,
        save_dir=cross_save_dir,
    )
    plt.show()
else:
    print("CNN-vs-best-embedding delta is not available until each dataset has both CNN and embedding results.")

cnn_embedding_delta.round(3)


### Representation Projection (t-SNE / UMAP)

Use the next cell to precompute and save every projection image into grouped directories. The widget after it only reads saved PNGs from the manifest, so switching between t-SNE, UMAP, models, and splits is fast.


In [ ]:
from notebooks.utils.projections import (
    ProjectionContext,
    load_projection_manifest,
    precompute_projection_plots,
)

projection_ctx = ProjectionContext(
    run_args=run_args,
    cnn_run_args=cnn_run_args,
    class_names=CLASS_NAMES,
    all_results=all_results,
    leaderboard=leaderboard,
    cnn_run_dir=CNN_RUN_DIR,
    run_name=RUN_NAME,
    save_dir=save_dir,
)
print(f"Projection root: {projection_ctx.projection_root}")


In [ ]:
# Precompute and save every projection used by the widget below.
# Outputs are grouped as:
#   PROJECTION_ROOT/images/<split>/<method>/<model>.png
#   PROJECTION_ROOT/data/<split>/<method>/<model>.npz
PROJECTION_MODELS = tuple(leaderboard["model_id"].tolist())
PROJECTION_SPLITS = ("hold-out", "validation")
PROJECTION_METHODS = ("tsne", "umap")
PROJECTION_MAX_PER_CLASS = 400
PROJECTION_SEED = 0
FORCE_RECOMPUTE_PROJECTIONS = False

projection_manifest = precompute_projection_plots(
    projection_ctx,
    model_ids=PROJECTION_MODELS,
    splits=PROJECTION_SPLITS,
    methods=PROJECTION_METHODS,
    max_per_class=PROJECTION_MAX_PER_CLASS,
    seed=PROJECTION_SEED,
    force=FORCE_RECOMPUTE_PROJECTIONS,
)

projection_manifest.groupby(["split", "method", "status"]).size().rename("count").reset_index()


In [ ]:
from IPython.display import Image as DisplayImage, display
from ipywidgets import Dropdown, interact

_saved_projection_manifest = load_projection_manifest(projection_ctx.index_path)
_ready_projection_manifest = _saved_projection_manifest[
    (_saved_projection_manifest["status"] == "ready")
    & (_saved_projection_manifest["png_exists"])
].copy()
if _ready_projection_manifest.empty:
    raise ValueError("No ready projection PNGs found. Run the precompute cell above first.")

_projection_options = [
    (
        f"{row['split']} | {row['method'].upper()} | {row['model_label']} ({row['model_id']})",
        str(idx),
    )
    for idx, row in _ready_projection_manifest.sort_values(
        ["split", "method", "model_label"]
    ).iterrows()
]


def show_saved_projection(projection: str):
    row = _ready_projection_manifest.loc[int(projection)]
    display(DisplayImage(filename=row["png_path"]))
    print(row["png_path"])


_ = interact(
    show_saved_projection,
    projection=Dropdown(options=_projection_options, description="Projection:"),
)
